# Ledger contributions per layer group, baseline vs ablation — padded (last-token) twin

The padded (last-token) twin of ledger_ablation_layers. The showcase §3 stacked
decomposition (ΔS = χ + quality + interference), summed over a chosen layer group only,
side by side: left the unablated padded sweep (block_representations_samples_padded), right
the run with a block's writes zeroed (data/results/ablate_padded_*). nanochat-d12 is
excluded (no padded data exists for it).

In [ ]:
import os, sys, importlib
if os.path.basename(os.getcwd()) == "analysis":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from analysis import experiments_lib as _lib
importlib.reload(_lib)
from analysis.experiments_lib import (plot_layer_contribution, plot_ledger_stack,
                                      grid_start, grid_show)

NB = {'pythia-160m-deduped': 12, 'pythia-410m-deduped': 24, 'pythia-1b-deduped': 16,
      'pythia-6.9b-deduped': 32, 'OLMo-2-0425-1B': 16, 'OLMo-2-1124-7B': 32}
CFG = lambda m: 'block_representations_samples_padded'
QUARTERS = {12: ['blk0-2', 'blk3-5', 'blk6-8', 'blk9-11', 'blk3-8'],
            16: ['blk0-3', 'blk4-7', 'blk8-11', 'blk12-15', 'blk4-11'],
            24: ['blk0-5', 'blk6-11', 'blk12-17', 'blk18-23', 'blk6-17'],
            32: ['blk0-7', 'blk8-15', 'blk16-23', 'blk24-31', 'blk8-23']}
CARRIER = {'pythia-1b-deduped': 'blk3', 'pythia-6.9b-deduped': 'blk3-4'}

def ledger_grid(model, rows, cols, title, savedir, figsize=None):
    """Grid of experiments_lib.plot_ledger_stack panels: rows = (label, block list),
    cols = (label, results dir). sharey='row' keeps each row's panels comparable without
    flattening single-layer rows against whole-quarter ones; all_xlabels keeps the token
    axis on every row. emb=False: the embedding line is only meaningful full-depth."""
    grid_start(ncols=len(cols), figsize=figsize or (6.5 * len(cols), 4 * len(rows)),
               sharex=True, sharey='row', all_xlabels=True, emb=False, zero_line=True,
               title=title, savedir=savedir)
    for i, (rlabel, layers) in enumerate(rows):
        for j, (clabel, cfg) in enumerate(cols):
            plot_ledger_stack(model, cfg, blocks=layers, title=f'{clabel} — {rlabel}',
                              ylabel='rank entropy' if j == 0 else None,
                              legend=7 if i == 0 and j == 0 else None)
    grid_show()

def ledger_fig(model, ablate):
    """3x2: rows = penultimate / final layer / last quarter, cols = baseline / ablated."""
    L = NB[model]
    rows = [(f'penultimate layer (blk{L - 2})', [L - 2]),
            (f'final layer (blk{L - 1})', [L - 1]),
            (f'last quarter (blk{3 * L // 4}-{L - 1})', list(range(3 * L // 4, L)))]
    ledger_grid(model, rows, [('baseline', CFG(model)), (f'− {ablate}', f'ablate_padded_{ablate}')],
                f'Ledger contributions per layer group — {model}, {ablate} writes zeroed',
                f'ledger_ablation_layers/{model}_{ablate}')

In [ ]:
# pythia-1b (16 blocks), blk3 (the carrier) ablated. Rows: the penultimate layer's
# contributions (blk14), the final layer's (blk15), the last quarter's (blk12-15).
# Left baseline, right ablated.
ledger_fig('pythia-1b-deduped', 'blk3')
# FIGURE B2?.?
# The pythia-6.9b blk3-4 carrier is now included too (added via the ABLATIONS loop below).

In [ ]:
# The other models: the carrier where one exists (pythia-6.9b: blk3-4), otherwise the
# first-quarter ablation — the early-blocks analog of blk3 (no other single-block runs).
# NB: no padded baseline for pythia-160m-deduped / pythia-410m-deduped — baseline panel will be empty for those two.
ABLATIONS = {'pythia-160m-deduped': 'blk0-2', 'pythia-410m-deduped': 'blk0-5',
             'pythia-6.9b-deduped': 'blk3-4', 'OLMo-2-0425-1B': 'blk0-3',
             'OLMo-2-1124-7B': 'blk0-7'}
for model, ablate in ABLATIONS.items():
    ledger_fig(model, ablate)

In [ ]:
# The per-block depth stacks (the experiments.ipynb ledger stacks), baseline vs ablated,
# per ledger term. The embedding entropy line only makes sense on the ΔS stack.
TERM_LABEL = {'delta_s': 'signed ΔS', 'quality': 'quality', 'interference': 'interference'}

def ledger_term_stacks(mdl, ablate, terms=('delta_s', 'quality', 'interference')):
    srcs = lambda cfg: [(cfg, (f'blk{l}', 'block_ledger'), f'blk {l}') for l in range(NB[mdl])]
    emb = lambda cfg: (cfg, ('blk0.attn.in', 'acts_centered'), 'embedding', 'matrix_entropy')
    for term in terms:
        grid_start(ncols=2, sharey=True,
                   title=f'{mdl} — rank-entropy ledger ({TERM_LABEL[term]}), baseline vs − {ablate}')
        for ttl, cfg in (('baseline', CFG(mdl)), (f'− {ablate}', f'ablate_padded_{ablate}')):
            plot_layer_contribution(mdl, srcs(cfg), yvar=term, normalize=False, title=ttl,
                                    alpha=0.85,
                                    **({'baseline_src': emb(cfg)} if term == 'delta_s' else {}))
        grid_show()

In [ ]:
# pythia-1b: the same three depth stacks, baseline vs the blk3 (carrier) ablation.
ledger_term_stacks('pythia-1b-deduped', 'blk3')

In [ ]:
# NB: no padded baseline for pythia-160m-deduped / pythia-410m-deduped — baseline panel will be empty for those two.
for model, nb in NB.items():
    ledger_term_stacks(model, QUARTERS[nb][0])

## OLMo-2: single mid-late layer + third quarter, baseline vs second-quarter ablation

In [ ]:
def ledger_pair(model, single):
    """2x2 ledger: rows = single layer / third quarter, cols = baseline / − second quarter."""
    L = NB[model]
    q2 = f'blk{L // 4}-{L // 2 - 1}'
    q3 = list(range(L // 2, 3 * L // 4))
    rows = [(f'blk{single}', [single]), (f'third quarter (blk{q3[0]}-{q3[-1]})', q3)]
    ledger_grid(model, rows, [('baseline', CFG(model)), (f'− {q2}', f'ablate_padded_{q2}')],
                f'Ledger contributions — {model}, baseline vs {q2} writes zeroed',
                f'ledger_ablation_layers/{model}_q2_ablation_single_q3', figsize=(13, 8))

In [ ]:
ledger_pair('OLMo-2-0425-1B', 10)

In [ ]:
ledger_pair('OLMo-2-1124-7B', 16)